# JAX Communication-Bit Curriculum

This notebook starts from the trained JAX `15x15` policy and progressively increases the number of writable communication bits. Increasing `WRITE_BITS` changes both the actor observation shape and the write-action head, so the notebook creates a warm-start checkpoint for each new bit width before training.

The warm start copies the old policy into the larger model: existing food/byte/hub/border/carrying inputs are preserved, new byte-bit input planes start with zero weights, and the larger write head is initialized by repeating the previous lower-bit write logits. That keeps the old behavior available while opening the larger communication alphabet.

In [ ]:
from pathlib import Path
import os
import sys

# These must be set before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.65")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

{
    "project_root": PROJECT_ROOT,
    "jax_preallocate": os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"],
    "jax_memory_fraction": os.environ["XLA_PYTHON_CLIENT_MEM_FRACTION"],
}


In [ ]:
import sys

try:
    import jax  # noqa: F401
    import jax.numpy as jnp  # noqa: F401
    from PIL import Image  # noqa: F401
    import tqdm  # noqa: F401
except ModuleNotFoundError as exc:
    missing = exc.name or "jax/notebook extras"
    raise ModuleNotFoundError(
        f"Missing {missing!r} in this notebook kernel ({sys.executable}). "
        "Install the JAX notebook extras from the repo root with: "
        f'{sys.executable} -m pip install -e ".[jax-cuda13,notebooks]"'
    ) from exc


In [ ]:
import importlib
import pickle
from types import SimpleNamespace

import jax
import jax.numpy as jnp
import numpy as np
from PIL import Image
from tqdm.auto import tqdm

from ant_byte_env import AntByteForagingEnv, MAX_WRITE_BITS, write_value_count
from ant_byte_env.vault import create_vault_entry
import train_mappo_jax
from train_mappo_jax_core import JaxMAPPOParams, LinearParams, init_adam_state

importlib.reload(train_mappo_jax)
from train_mappo_jax import (
    build_actor_observations,
    build_central_observations,
    flatten_agent_actions,
    get_action_and_value,
    main,
    save_checkpoint,
)


## Curriculum Settings

`BIT_STAGES` intentionally begins above `1`: the base checkpoint is already the trained 1-bit `15x15` policy. Edit `GLOBAL_UPDATE_CAP` and `BIT_STAGES` before running if you want a shorter smoke run or a more aggressive curriculum.

In [ ]:
CHECKPOINT_DIR = PROJECT_ROOT / "checkpoints"
COMM_CHECKPOINT_DIR = CHECKPOINT_DIR / "jax_communication_curriculum"
COMM_CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

BASE_CHECKPOINT = CHECKPOINT_DIR / "jax_mappo_forage_stage1_15x15.pkl"
BIT_STAGES = [2, 3, 5, 8]

NUM_ENVS = 16
NUM_STEPS = 80
UPDATE_TIMESTEPS = NUM_ENVS * NUM_STEPS
GLOBAL_UPDATE_CAP = 100
ROLLOUT_TILE_SIZE = 32

if not BASE_CHECKPOINT.exists():
    raise FileNotFoundError(f"Train or restore the base checkpoint first: {BASE_CHECKPOINT}")
if any(bits <= 1 or bits > MAX_WRITE_BITS for bits in BIT_STAGES):
    raise ValueError(f"BIT_STAGES must contain integers from 2 to {MAX_WRITE_BITS}.")
if list(BIT_STAGES) != sorted(BIT_STAGES):
    raise ValueError("BIT_STAGES must be increasing.")

print(f"JAX device: {jax.devices()[0]}")
print(f"Base checkpoint: {BASE_CHECKPOINT}")
print(f"Communication bit stages: {BIT_STAGES}")


## Warm-Start Adapter

A direct `--load-model` fails across bit widths because the first actor layer and the write head have different shapes. These helpers rewrite a checkpoint into the target bit width while preserving the old policy as much as possible.

In [ ]:
def load_raw_checkpoint(checkpoint_path):
    with Path(checkpoint_path).open("rb") as checkpoint_file:
        checkpoint = pickle.load(checkpoint_file)
    checkpoint["params"] = jax.tree_util.tree_map(jnp.asarray, checkpoint["params"])
    checkpoint["opt_state"] = jax.tree_util.tree_map(jnp.asarray, checkpoint["opt_state"])
    return checkpoint


def checkpoint_write_bits(checkpoint):
    return int(checkpoint["args"].get("write_bits", 1))


def checkpoint_actor_radius(checkpoint):
    return int(checkpoint["args"].get("actor_vision_radius", 2))


def actor_obs_dim_for_bits(*, write_bits, actor_vision_radius):
    patch_size = (2 * actor_vision_radius + 1) ** 2
    return patch_size * (write_bits + 3) + 1


def expand_actor_input_layer(layer, *, old_bits, target_bits, actor_vision_radius):
    if target_bits < old_bits:
        raise ValueError("target_bits must not be smaller than old_bits.")

    old_weight = jnp.asarray(layer.weight)
    old_bias = jnp.asarray(layer.bias)
    patch_size = (2 * actor_vision_radius + 1) ** 2
    expected_old_dim = actor_obs_dim_for_bits(
        write_bits=old_bits,
        actor_vision_radius=actor_vision_radius,
    )
    target_dim = actor_obs_dim_for_bits(
        write_bits=target_bits,
        actor_vision_radius=actor_vision_radius,
    )
    if old_weight.shape[0] != expected_old_dim:
        raise ValueError(
            f"Expected actor input dim {expected_old_dim}, got {old_weight.shape[0]}."
        )

    new_weight = jnp.zeros((target_dim, old_weight.shape[1]), dtype=old_weight.dtype)
    old_food = slice(0, patch_size)
    old_bits_slice = slice(patch_size, patch_size * (1 + old_bits))
    old_hub = slice(patch_size * (1 + old_bits), patch_size * (2 + old_bits))
    old_border = slice(patch_size * (2 + old_bits), patch_size * (3 + old_bits))

    new_food = old_food
    new_bits_slice = old_bits_slice
    new_hub = slice(patch_size * (1 + target_bits), patch_size * (2 + target_bits))
    new_border = slice(patch_size * (2 + target_bits), patch_size * (3 + target_bits))

    new_weight = new_weight.at[new_food, :].set(old_weight[old_food, :])
    new_weight = new_weight.at[new_bits_slice, :].set(old_weight[old_bits_slice, :])
    new_weight = new_weight.at[new_hub, :].set(old_weight[old_hub, :])
    new_weight = new_weight.at[new_border, :].set(old_weight[old_border, :])
    new_weight = new_weight.at[-1, :].set(old_weight[-1, :])
    return LinearParams(weight=new_weight, bias=old_bias)


def expand_write_head(layer, *, old_bits, target_bits):
    old_weight = jnp.asarray(layer.weight)
    old_bias = jnp.asarray(layer.bias)
    old_count = write_value_count(old_bits)
    target_count = write_value_count(target_bits)
    if old_weight.shape[-1] != old_count:
        raise ValueError(f"Expected {old_count} old write logits, got {old_weight.shape[-1]}.")
    source_indices = jnp.arange(target_count) % old_count
    return LinearParams(
        weight=old_weight[:, source_indices],
        bias=old_bias[source_indices],
    )


def expand_params_for_write_bits(params, *, old_bits, target_bits, actor_vision_radius):
    if target_bits == old_bits:
        return params
    return JaxMAPPOParams(
        actor_body=(
            expand_actor_input_layer(
                params.actor_body[0],
                old_bits=old_bits,
                target_bits=target_bits,
                actor_vision_radius=actor_vision_radius,
            ),
            params.actor_body[1],
        ),
        move_head=params.move_head,
        write_head=expand_write_head(
            params.write_head,
            old_bits=old_bits,
            target_bits=target_bits,
        ),
        critic_body=params.critic_body,
        value_head=params.value_head,
    )


def write_warm_start_checkpoint(source_checkpoint_path, *, target_bits):
    source_checkpoint_path = Path(source_checkpoint_path)
    checkpoint = load_raw_checkpoint(source_checkpoint_path)
    old_bits = checkpoint_write_bits(checkpoint)
    actor_vision_radius = checkpoint_actor_radius(checkpoint)
    if target_bits < old_bits:
        raise ValueError(f"Cannot warm-start from {old_bits} bits down to {target_bits} bits.")
    if target_bits == old_bits:
        return source_checkpoint_path

    target_actor_obs_dim = actor_obs_dim_for_bits(
        write_bits=target_bits,
        actor_vision_radius=actor_vision_radius,
    )
    params = expand_params_for_write_bits(
        checkpoint["params"],
        old_bits=old_bits,
        target_bits=target_bits,
        actor_vision_radius=actor_vision_radius,
    )
    opt_state = init_adam_state(params)
    target_args = dict(checkpoint["args"])
    target_args["write_bits"] = target_bits
    target_args["load_model"] = str(source_checkpoint_path)
    warm_start_path = COMM_CHECKPOINT_DIR / (
        f"warm_start_{old_bits}_to_{target_bits}_bits_from_{source_checkpoint_path.stem}.pkl"
    )
    target_args["save_model"] = str(warm_start_path)
    save_checkpoint(
        warm_start_path,
        params=params,
        opt_state=opt_state,
        args=SimpleNamespace(**target_args),
        central_obs_dim=int(checkpoint["central_obs_dim"]),
        actor_obs_dim=target_actor_obs_dim,
        run_name=f"warm_start_{old_bits}_to_{target_bits}_bits",
        metrics=dict(checkpoint.get("metrics", {})),
    )
    return warm_start_path


base_checkpoint = load_raw_checkpoint(BASE_CHECKPOINT)
{
    "base_write_bits": checkpoint_write_bits(base_checkpoint),
    "base_actor_obs_dim": int(base_checkpoint["actor_obs_dim"]),
    "base_write_head_shape": tuple(base_checkpoint["params"].write_head.weight.shape),
}


## Train Bit Stages

Each stage uses the previous checkpoint as the source, expands it to the new bit width, then resumes training on the `15x15` task. The default `GLOBAL_UPDATE_CAP` is deliberately modest; increase it for real runs.

In [ ]:
base_args = SimpleNamespace(**base_checkpoint["args"])
COMMON_ARGS = [
    "--num-envs", str(NUM_ENVS),
    "--num-steps", str(NUM_STEPS),
    "--num-minibatches", str(base_args.num_minibatches),
    "--update-epochs", str(base_args.update_epochs),
    "--width", str(base_args.width),
    "--height", str(base_args.height),
    "--obs-width", str(base_args.obs_width),
    "--obs-height", str(base_args.obs_height),
    "--actor-vision-radius", str(base_args.actor_vision_radius),
    "--num-ants", str(base_args.num_ants),
    "--food-count", str(base_args.food_count),
    "--food-sources", str(base_args.food_sources),
    "--cookie-distance", str(base_args.cookie_distance),
    "--max-steps", str(base_args.max_steps),
    "--pickup-bonus", str(base_args.pickup_bonus),
    "--distance-bonus", str(base_args.distance_bonus),
    "--hidden-size", str(base_args.hidden_size),
    "--seed", str(base_args.seed),
    "--quiet",
]
if base_args.random_food:
    COMMON_ARGS.append("--random-food")
if base_args.random_hub:
    COMMON_ARGS.append("--random-hub")
COMMON_ARGS


In [ ]:
stage_metrics = []
stage_checkpoint_paths = []
previous_checkpoint = BASE_CHECKPOINT

for target_bits in BIT_STAGES:
    warm_start_path = write_warm_start_checkpoint(
        previous_checkpoint,
        target_bits=target_bits,
    )
    checkpoint_path = COMM_CHECKPOINT_DIR / f"jax_mappo_15x15_{target_bits}_bits.pkl"
    print(f"Training communication stage: {target_bits} writable bits")
    print(f"Warm start: {warm_start_path}")
    update_iterator = tqdm(
        range(1, GLOBAL_UPDATE_CAP + 1),
        total=GLOBAL_UPDATE_CAP,
        desc=f"{target_bits} bits",
        bar_format="{desc}: {n_fmt}/{total_fmt} updates |{bar}| {elapsed}<{remaining} {postfix}",
        leave=True,
    )

    def record_progress(update_index, total_updates, train_metrics):
        del total_updates
        update_iterator.update(1)
        update_iterator.set_postfix(
            loss=f"{train_metrics['loss']:.3f}",
            ret=f"{train_metrics['episode_return']:.3f}",
        )
        stage_metrics.append(
            {
                "write_bits": target_bits,
                **train_metrics,
                "stage_update": update_index,
                "global_update_cap": GLOBAL_UPDATE_CAP,
                "checkpoint": str(checkpoint_path),
                "warm_start_checkpoint": str(warm_start_path),
            }
        )

    train_args = [
        *COMMON_ARGS,
        "--write-bits", str(target_bits),
        "--total-timesteps", str(UPDATE_TIMESTEPS * GLOBAL_UPDATE_CAP),
        "--load-model", str(warm_start_path),
        "--save-model", str(checkpoint_path),
    ]
    try:
        final_train_metrics = main(train_args, progress_callback=record_progress)
    finally:
        update_iterator.close()

    stage_checkpoint_paths.append(checkpoint_path)
    previous_checkpoint = checkpoint_path
    print(f"Saved {target_bits}-bit checkpoint to {checkpoint_path}")

FINAL_COMMUNICATION_CHECKPOINT = previous_checkpoint
{
    "stage_checkpoint_paths": stage_checkpoint_paths,
    "final_checkpoint": FINAL_COMMUNICATION_CHECKPOINT,
    "final_train_metrics": final_train_metrics,
}


## Optional Render and Vault

Run this after training if you want rollout GIFs for the communication curriculum. It uses Pillow rather than ffmpeg so it does not fork a subprocess from the JAX kernel.

In [ ]:
def _sample_hub_position(args, rng):
    if getattr(args, "random_hub", False):
        return (
            int(rng.integers(0, args.width)),
            int(rng.integers(0, args.height)),
        )
    return (args.width // 2, args.height // 2)


def build_reset_options(args, *, seed=None):
    rng = np.random.default_rng(seed)
    hub = _sample_hub_position(args, rng)
    if getattr(args, "random_food", False):
        return {"hub_pos": hub}
    return {"hub_pos": hub}


def obs_to_jax_batch(obs):
    return {key: jnp.asarray(value[None, ...]) for key, value in obs.items()}


def save_rollout_gif(path, frames, *, fps):
    duration_ms = max(1, round(1000 / fps))
    pil_frames = [Image.fromarray(np.asarray(frame, dtype=np.uint8)).convert("RGB") for frame in frames]
    pil_frames[0].save(
        path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,
    )


def render_policy_rollout(checkpoint_path):
    checkpoint = load_raw_checkpoint(checkpoint_path)
    saved_args = SimpleNamespace(**checkpoint["args"])
    params = checkpoint["params"]
    write_bits = int(saved_args.write_bits)
    action_key = jax.random.PRNGKey(saved_args.seed + 100_000 + write_bits)
    env = AntByteForagingEnv(
        width=saved_args.width,
        height=saved_args.height,
        num_ants=saved_args.num_ants,
        food_count=saved_args.food_count,
        food_source_count=saved_args.food_sources,
        max_steps=saved_args.max_steps,
        random_food=saved_args.random_food,
        render_mode="rgb_array",
        tile_size=ROLLOUT_TILE_SIZE,
        write_bits=write_bits,
    )
    frames = []
    try:
        obs, info = env.reset(
            seed=saved_args.seed,
            options=build_reset_options(saved_args, seed=saved_args.seed),
        )
        first_frame = env.render()
        if first_frame is not None:
            frames.append(first_frame)
        for _ in tqdm(range(saved_args.max_steps), desc=f"{checkpoint_path.stem} rollout", leave=False):
            obs_batch = obs_to_jax_batch(obs)
            central_obs = build_central_observations(
                obs_batch,
                food_scale=saved_args.food_count,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            actor_obs = build_actor_observations(
                obs_batch,
                food_scale=saved_args.food_count,
                actor_vision_radius=saved_args.actor_vision_radius,
                write_bits=write_bits,
                obs_width=saved_args.obs_width,
                obs_height=saved_args.obs_height,
            )
            action_key, step_key = jax.random.split(action_key)
            joint_actions, _, _, _ = get_action_and_value(
                params,
                actor_obs,
                central_obs,
                step_key,
                deterministic=False,
            )
            env_action = np.asarray(flatten_agent_actions(joint_actions))[0]
            obs, reward, terminated, truncated, info = env.step(env_action)
            frame = env.render()
            if frame is not None:
                frames.append(frame)
            if terminated or truncated:
                break
    finally:
        env.close()
    if not frames:
        raise RuntimeError(f"No frames were rendered for {checkpoint_path}.")
    rollout_path = Path(checkpoint_path).with_name(f"{Path(checkpoint_path).stem}_rollout.gif")
    save_rollout_gif(rollout_path, frames, fps=AntByteForagingEnv.metadata["render_fps"])
    return rollout_path


In [ ]:
policy_checkpoint_paths = [
    COMM_CHECKPOINT_DIR / f"jax_mappo_15x15_{bits}_bits.pkl"
    for bits in BIT_STAGES
]
missing_checkpoints = [path for path in policy_checkpoint_paths if not path.exists()]
if missing_checkpoints:
    missing = "\n".join(str(path) for path in missing_checkpoints)
    raise FileNotFoundError(f"Train the missing communication stages before rendering:\n{missing}")

rollout_paths = [
    render_policy_rollout(path)
    for path in tqdm(policy_checkpoint_paths, desc="rendering communication policies")
]
vault_entry_path = create_vault_entry(
    vault_dir=PROJECT_ROOT / "vault",
    title="JAX MAPPO communication-bit curriculum",
    description="Rollout GIFs for 15x15 JAX MAPPO policies trained with progressively larger writable communication alphabets.",
    assets=rollout_paths,
    metadata={
        "base_checkpoint": str(BASE_CHECKPOINT),
        "bit_stages": BIT_STAGES,
        "checkpoint_paths": [str(path) for path in policy_checkpoint_paths],
        "rollout_paths": [str(path) for path in rollout_paths],
        "global_update_cap": GLOBAL_UPDATE_CAP,
    },
)
{
    "rollout_paths": rollout_paths,
    "vault_entry_path": vault_entry_path,
}
